# 🚀 Walkability Semantic Segmentation: Google Colab Training

This notebook runs the complete 4-class semantic segmentation training pipeline for **BiSeNetV2** using Google Colab's free **Tesla T4 GPU (15-16 GB VRAM)**.

### ⚠️ Step 0: Ensure GPU Acceleration is Enabled
1. Go to the top menu: **Runtime** -> **Change runtime type**
2. Under *Hardware accelerator*, select **T4 GPU**
3. Click **Save**

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Mount Google Drive
Mount your Google Drive to access the dataset zip files and to automatically persist trained checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository & Checkout Training Branch

In [ ]:
import os
%cd /content

if not os.path.exists("/content/Capstone-Project"):
    !git clone https://github.com/dev-june05/Capstone-Project.git

%cd /content/Capstone-Project
!git checkout divyansh/model-training
!git pull origin divyansh/model-training

## 4. Install Dependencies

In [ ]:
!pip install -r requirements.txt
!pip install albumentations

## 5. Unzip Cityscapes from Google Drive to Local Colab SSD
> **Tip:** Unzipping directly onto Colab's local NVMe SSD (`/content/Capstone-Project/data/cityscapes/`) takes only 1-2 minutes and makes training much faster than reading directly from Google Drive over network I/O.

Upload `leftImg8bit_trainvaltest.zip` and `gtFine_trainvaltest.zip` to your Google Drive (e.g. inside a folder named `Cityscapes`).

In [ ]:
import os
from pathlib import Path

# Ensure target directory exists
dest_dir = "/content/Capstone-Project/data/cityscapes"
os.makedirs(dest_dir, exist_ok=True)

# Set this to the folder in Google Drive where you placed the zip files
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Cityscapes"

leftimg_zip = os.path.join(DRIVE_DATASET_DIR, "leftImg8bit_trainvaltest.zip")
gtfine_zip = os.path.join(DRIVE_DATASET_DIR, "gtFine_trainvaltest.zip")

if os.path.exists(leftimg_zip):
    print("Unzipping leftImg8bit images...")
    !unzip -q "{leftimg_zip}" -d "{dest_dir}"
    print("Images unzipped successfully!")
else:
    print(f"❌ Not found: {leftimg_zip}. Please check your Google Drive folder name.")

if os.path.exists(gtfine_zip):
    print("Unzipping gtFine masks...")
    !unzip -q "{gtfine_zip}" -d "{dest_dir}"
    print("Masks unzipped successfully!")
else:
    print(f"❌ Not found: {gtfine_zip}. Please check your Google Drive folder name.")

## 6. Verify Dataset Loading

In [ ]:
# Verify that CityscapesDataset can detect the images and masks
from utils.dataset import CityscapesDataset

train_ds = CityscapesDataset("data/cityscapes", split="train")
val_ds = CityscapesDataset("data/cityscapes", split="val")
print(f"✅ Training samples detected: {len(train_ds)} (expected: 2975)")
print(f"✅ Validation samples detected: {len(val_ds)} (expected: 500)")

## 7. Start Training (BiSeNetV2 on T4 GPU)
The Colab configuration uses `batch_size: 8` and `num_workers: 4` to fully leverage the 15 GB VRAM and multi-core CPU of Google Colab.

In [ ]:
!python scripts/train.py --config configs/train_colab.yaml

## 8. Backup Best Checkpoints to Google Drive
Run this to guarantee your weights are saved to your Google Drive even after the Colab session disconnects.

In [ ]:
backup_dir = "/content/drive/MyDrive/capstone_checkpoints"
os.makedirs(backup_dir, exist_ok=True)

!cp -r checkpoints/* "{backup_dir}/"
print(f"✅ All checkpoints copied to Google Drive: {backup_dir}")

## 9. Evaluate the Best Model
Compute mIoU, per-class IoU, Obstacle Recall, and Missed Obstacle Rate on the validation split.

In [ ]:
!python scripts/evaluate.py --config configs/train_colab.yaml --checkpoint checkpoints/best.pth --split val